<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">MODULE 8: VIEWS AND MATERIALIZED VIEWS</div><div style="color:#17212b;font-size:30px;font-weight:750">Turn a stable business query into a logical view, then use a materialized view to precompute a repeated aggregate</div><p style="color:#475569;line-height:1.7">Run in order against a dedicated Level 1 course database. Every result is rendered as a table and every write is scoped to this module's objects.</p></div>

## Boundary

This lab never changes the Level 1 source tables. It creates or replaces only objects with the `_l2` suffix.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP VIEW IF EXISTS orders_service_view_l2")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_daily_mv_l2")
lab.execute("""
CREATE VIEW orders_service_view_l2 AS
SELECT order_date, customer_id, order_amount, data_source
FROM orders_imported
WHERE order_amount > 0
""")
lab.sql("SELECT * FROM orders_service_view_l2 ORDER BY order_date, customer_id LIMIT 10", title="Stable service view")

In [ ]:
lab.execute("""
CREATE MATERIALIZED VIEW orders_daily_mv_l2
BUILD IMMEDIATE
REFRESH AUTO ON MANUAL
DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1")
AS
SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_imported
GROUP BY order_date
""")
lab.sql("SHOW MATERIALIZED VIEWS FROM dw_course_l1_demo", title="Materialized-view metadata")

In [ ]:
lab.sql("""
SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_service_view_l2
GROUP BY order_date
ORDER BY order_date
""", title="Canonical daily metric")
lab.sql("""
EXPLAIN
SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_imported
GROUP BY order_date
ORDER BY order_date
""", title="Rewrite evidence for the repeated metric", final=True)

## Takeaway

Compare the result with the business grain stated in the lesson. A successful SQL statement is not by itself evidence that the model, metric, access boundary, or consumer contract is correct.